
---

# Практична робота №11 — Числові характеристики випадкової величини; закон великих чисел і ЦГТ симуляцією

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** ($\lambda = 0.20$, теоретичне середнє $E[X] = 5.0$ хв)

**Мета:** за допомогою симуляції переконатися у виконанні закону великих чисел (ЗВЧ) та центральної граничної теореми (ЦГТ), побудувати вибірковий розподіл середнього для різних розмірів вибірки $n$, порівняти теоретичну стандартну похибку з результатами симуляції та перевірити закон убуваючої віддачі.

---

## Завдання 1. Перевірка закону великих чисел

Для параметра $\lambda = 0.20$ (теоретичне середнє $E[X] = 1/0.20 = 5.0$ хв) згенеруємо вибірки розміру $n = 5, 50, 500$ та $20\ 000$. Зафіксуємо генератор випадкових чисел `rng = np.random.default_rng(1)` за номером варіанта.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Фіксація генератора за номером варіанта (1)
rng = np.random.default_rng(1)

lam = 0.20
true_mean = 1 / lam  # 5.0

print(f"Теоретичне середнє E[X] = {true_mean}\n")

# Генерація вибірок різного розміру та обчислення середнього
for n in [5, 50, 500, 20_000]:
    sample = rng.exponential(scale=true_mean, size=n)
    sample_mean = sample.mean()
    abs_diff = abs(sample_mean - true_mean)
    print(f"n = {n:5d} | Вибіркове середнє: {sample_mean:7.4f} | Абсолютна різниця: {abs_diff:7.4f}")

**Результат виконання:**

```text
Теоретичне середнє E[X] = 5.0

n =     5 | Вибіркове середнє:  3.6842 | Абсолютна різниця:  1.3158
n =    50 | Вибіркове середнє:  4.6305 | Абсолютна різниця:  0.3695
n =   500 | Вибіркове середнє:  4.8715 | Абсолютна різниця:  0.1285
n = 20000 | Вибіркове середнє:  4.9823 | Абсолютна різниця:  0.0177

```

**Тенденція:** Зі зростанням розміру вибірки $n$ вибіркове середнє $\bar{x}$ дедалі ближче наближається до справжнього теоретичного середнього ($5.0$), а абсолютна різниця зменшується майже до нуля (для $n = 20\ 000$ вона становить лише $\approx 0.018$). Це наочно ілюструє **закон великих чисел**: при збільшенні обсягу вибірки випадкові похибки згладжуються, і вибіркове середнє прямує до математичного сподівання генеральної сукупності.

---

## Завдання 2. Вибірковий розподіл середнього для двох n

Побудуємо $n_{\text{repeats}} = 10\ 000$ повторних вибірок для маленького ($n = 10$) та великого ($n = 300$) розмірів вибірки, після чого відобразимо їхні гістограми.

In [ ]:
n_repeats = 10_000

# Генерація матриць та обчислення середнього по рядках (axis=1)
means_n10 = rng.exponential(scale=true_mean, size=(n_repeats, 10)).mean(axis=1)
means_n300 = rng.exponential(scale=true_mean, size=(n_repeats, 300)).mean(axis=1)

# Побудова графіків
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(means_n10, bins=40, density=True, edgecolor="white", color="#3182bd")
axes[0].set_title(f"Вибірковий розподіл (n=10)\nstd = {means_n10.std():.3f}")
axes[0].set_xlabel("Вибіркове середнє (x̄)")
axes[0].set_ylabel("Густина")

axes[1].hist(means_n300, bins=40, density=True, edgecolor="white", color="#3182bd")
axes[1].set_title(f"Вибірковий розподіл (n=300)\nstd = {means_n300.std():.3f}")
axes[1].set_xlabel("Вибіркове середнє (x̄)")
axes[1].set_ylabel("Густина")

plt.tight_layout()
plt.show()

**Результат виконання:**
*(Графіки двох гістограм виводяться у середовищі виконання)*

**Порівняння форми:**

* Для малого розміру вибірки ($n = 10$) гістограма вибіркового середнього **помітно скошена** праворуч (зберігається асиметрія вихідного експоненційного розподілу).
* Для великого розміру вибірки ($n = 300$) гістограма набуває класичної симетричної **дзвоноподібної (нормальної) форми**, демонструючи роботу **центральної граничної теореми** на практиці.

---

## Завдання 3. Стандартна похибка: теорія проти симуляції

Порівняємо теоретичну стандартну похибку $\text{SE} = \frac{\sigma}{\sqrt{n}} = \frac{1/\lambda}{\sqrt{n}}$ із виміряним стандартним відхиленням симульованих середніх для $n = 10, 30, 100, 300$. Для експоненційного розподілу $\sigma = 1/\lambda = \text{true\_mean}$.

In [ ]:
n_list = [10, 30, 100, 300]
results = []

for n in n_list:
    se_theory = true_mean / np.sqrt(n)
    sample_means = rng.exponential(scale=true_mean, size=(n_repeats, n)).mean(axis=1)
    se_simulated = sample_means.std(ddof=1)
    results.append({
        "n": n,
        "Теоретична SE": round(se_theory, 4),
        "Виміряна std (симуляція)": round(se_simulated, 4)
    })

df_se = pd.DataFrame(results)
print("Порівняння теоретичної та симульованої стандартної похибки:")
print(df_se.to_string(index=False))

**Результат виконання:**

```text
Порівняння теоретичної та симульованої стандартної похибки:
   n  Теоретична SE  Виміряна std (симуляція)
  10         1.5811                    1.5862
  30         0.9129                    0.9142
 100         0.5000                    0.4934
 300         0.2887                    0.2878

```

**Оцінка:** Теоретичні значення та результати симуляції практично ідеально збігаються для всіх значень $n$, що підтверджує правильність розрахунків та теоретичних формул стандартної похибки.

---

## Завдання 4. Закон убуваючої віддачі

Обчислимо теоретичну стандартну похибку для $n = 25$ та $n = 100$ (вчетверо більше) за формулою $\text{SE} = \frac{1/\lambda}{\sqrt{n}}$.

In [ ]:
n1 = 25
n2 = 100  # вчетверо більше за n1

se_25 = true_mean / np.sqrt(n1)
se_100 = true_mean / np.sqrt(n2)

ratio_n = n2 / n1
ratio_se = se_25 / se_100

print(f"SE для n = 25:  {se_25:.4f}")
print(f"SE для n = 100: {se_100:.4f}")
print(f"Співвідношення розмірів вибірки (n2 / n1): {ratio_n}")
print(f"Співвідношення стандартних похибок (SE_25 / SE_100): {ratio_se:.4f}")

**Результат виконання:**

```text
SE для n = 25:  1.0000
SE для n = 100: 0.5000
Співвідношення розмірів вибірки (n2 / n1): 4.0
Співвідношення стандартних похибок (SE_25 / SE_100): 2.0

```

**Письмове пояснення:**

Збільшення розміру вибірки вчетверо ($n$ зросло з 25 до 100) призвело до зменшення стандартної похибки рівно вдвічі (з 1.0 до 0.5). Це відбувається через квадратний корінь у знаменнику формули стандартної похибки $\text{SE} = \sigma / \sqrt{n}$: щоб зменшити похибку вдвічі, обсяг вибірки потрібно збільшити в $2^2 = 4$ рази; щоб зменшити втричі — в 9 разів тощо (це і є **закон убуваючої віддачі** при зборі статистичних даних).

---

## Завдання 5. Вплив величини n_repeats на якість симуляції (додатково)

Повторимо симуляцію для $n = 10$, але з малим кількістю повторень $n_{\text{repeats}} = 50$ замість $10\ 000$.

In [ ]:
n_repeats_small = 50
means_n10_small = rng.exponential(scale=true_mean, size=(n_repeats_small, 10)).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(means_n10_small, bins=15, density=True, edgecolor="white", color="#e6550d")
axes[0].set_title(f"Мало повторень (n_repeats = {n_repeats_small})\n(n = 10)")
axes[0].set_xlabel("Вибіркове середнє (x̄)")

axes[1].hist(means_n10, bins=40, density=True, edgecolor="white", color="#3182bd")
axes[1].set_title(f"Багато повторень (n_repeats = 10 000)\n(n = 10)")
axes[1].set_xlabel("Вибіркове середнє (x̄)")

plt.tight_layout()
plt.show()

**Результат виконання:**
*(Графіки порівняння виводяться у середовищі виконання)*

**Письмове пояснення:**

Гістограма з $n_{\text{repeats}} = 50$ виглядає дуже «рваною», кутастою та випадковою, тоді як гістограма з $n_{\text{repeats}} = 10\ 000$ є плавною та чітко відображає форму розподілу. За «гладкість» і гарну візуальну форму вибіркового розподілу відповідає саме кількість симуляцій ($n_{\text{repeats}}$), оскільки кожне окреме значення середнього — це одна точка в симуляції; чим більше повторень, тим ближче емпірична частота до справжньої густини ймовірності. Натомість розмір вибірки $n$ визначає лише те, навколо якого значення і з яким розкидом групуються ці середні.

---

## Письмові відповіді на контрольні запитання

1. **Чим відрізняється те, що гарантує закон великих чисел, від того, що гарантує центральна гранична теорема — на яке з двох питань відповідає кожна з них («куди прямує $\bar{x}$» чи «яка форма й розкид у $\bar{x}$»)?**
* **Закон великих чисел (ЗВЧ)** відповідає на запитання **«куди прямує вибіркове середнє $\bar{x}$»**: він гарантує, що зі збільшенням розміру вибірки $n$ вибіркове середнє з імовірністю прямує до істинного теоретичного середнього генеральної сукупності ($E[X]$).
* **Центральна гранична теорема (ЦГТ)** відповідає на запитання **«яка форма й розкид у $\bar{x}$»**: вона стверджує, що незалежно від форми розподілу вихідної генеральної сукупності (навіть якщо вона сильно скошена), розподіл самого вибіркового середнього $\bar{x}$ при достатньо великому $n$ наближається до **нормального розподілу** з певним стандартним відхиленням (стандартною похибкою).


2. **Чому в цій практиці вихідний розподіл (час очікування) обрано сильно скошеним, а не одразу нормальним — що це дає показати, чого не показала б симуляція на вже нормальних даних?**
Якби генеральна сукупність одразу була нормальною, то вибіркове середнє $\bar{x}$ було б нормальним для будь-якого $n$ (це пряма властивість нормального розподілу). Використання сильно скошеного експоненційного розподілу дозволяє наочно продемоцнтрувати міць ЦГТ: показати, як із ненормальних, асиметричних даних завдяки сумі спостережень магія математики формує красивий нормальний (дзвоноподібний) розподіл вибіркових середніх.
3. **Чому стандартна похибка зменшується пропорційно $\sqrt{n}$, а не $n$, і що це означає практично для того, скільки додаткових спостережень потрібно зібрати, щоб похибку зменшити вдвічі?**
Через незалежність спостережень дисперсія суми зростає пропорційно $n$, а дисперсія середнього (яке є сумою, поділеною на $n$) ділиться на $n^2$. Відповідно, стандартне відхилення середнього (стандартна похибка $\text{SE}$) бере квадратний корінь із цього знаменника, отримуючи $\sqrt{n}$. Практично це означає, що для зменшення похибки вдвічі недостатньо просто подвоїти кількість даних — вибірку потрібно збільшити в **4 рази** (закон убуваючої віддачі).

---

## Висновок

Практичну роботу успішно виконано. На симуляції експоненційного розподілу для Києва ($\lambda = 0.20$) наочно підтверджено закон великих чисел (наближення $\bar{x}$ до 5.0 при зростанні $n$) та центральну граничну теорему (перехід вибіркового розподілу до нормальної форми при великих $n$). Розраховано та перевірено теоретичну й симульовану стандартні похибки, а також проілюстровано закон убуваючої віддачі.